###Agente Farmacêutico Explicador de Bulas (RAG)

####Dependências

In [0]:
pip install langchain==0.1.20 langchain-community==0.0.38 langchain-openai==0.1.7 chromadb pypdf

In [0]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai
!pip install langchain==0.1.20 langchain-community==0.0.38 langchain-openai==0.1.7
!pip install pypdf
!pip install chromadb

####Bibliotecas

In [0]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

#### Definição do Problema
Bulas farmacêuticas possuem:

Linguagem técnica
Grande volume de texto
Informações sensíveis
O objetivo do agente é interpretar corretamente as bulas, respondendo perguntas somente com base nos documentos, evitando qualquer tipo de alucinação.

In [0]:

# Lista com os caminhos dos arquivos PDF das bulas
caminhos_bulas = [
    "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/dipirona.pdf",
    "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/paracetamol.pdf"
]

# Lista que armazenará todos os documentos carregados
documentos = []

# Percorre cada bula
for caminho in caminhos_bulas:

    # Cria o loader do PDF
    loader = PyPDFLoader(caminho)

    # Carrega o conteúdo do PDF
    docs = loader.load()

    # Adiciona o nome do medicamento como metadado
    for doc in docs:
        doc.metadata["medicamento"] = caminho.split("/")[-1].replace(".pdf", "")

    # Adiciona os documentos à lista principal
    documentos.extend(docs)

# Exibe a quantidade total de páginas carregadas
len(documentos)

In [0]:
print(documentos)

#### Extração, Limpeza e Chunking

In [0]:
####Dividi o conteúdo das bulas em blocos menores, preservando contexto por meio de sobreposição.

# Cria o objeto responsável pelo chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      
    chunk_overlap=150  
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documentos)

# Quantidade total de chunks gerados
len(chunks)

In [0]:
chunks

#### Enriquecimento com Metadados
- Filtragem
- Explicabilidade
- Melhor relevância na recuperação

In [0]:
# Percorre cada chunk para classificar semanticamente seu conteúdo
for chunk in chunks:

    # Normaliza o texto para facilitar as verificações
    texto = chunk.page_content.lower()
    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"
    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"
    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"
    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"
    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"
    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"
    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"
    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"
    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"
    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"

    else:
        chunk.metadata["categoria"] = "geral"

In [0]:
import random

# Seleciona dois chunks aleatórios
chunks_aleatorios = random.sample(chunks, 2)

# Imprime os metadados e um trecho do conteúdo
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Chunk Aleatório {i} ---")
    print("Metadados:")
    print(chunk.metadata)
    print("\nConteúdo (início):")
    print(chunk.page_content[:300])